In [50]:
#!pip install transformers
#!pip install torch torchvision
#!pip install scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [1]:
# load libraries
import pandas as pd
import json
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import RobertaTokenizerFast
from transformers import RobertaForTokenClassification
import torch, torchvision
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm

In [16]:
# load the tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

# function that creates BIO-tags for text
def tokenize_and_align_labels(text, entities, label_to_id):

    # tokenize and get offsets
    encoding = tokenizer(text, return_offsets_mapping=True, truncation=True)
    # initialize label list with as many "O" labels as there are tokens
    labels = ["O"] * len(encoding.offset_mapping)
    
    # loop through all spans, get start and end position as well as the label
    for ent in entities:
        start, end = ent["start"], ent["end"]
        ent_label = ent["labels"]
        
        # loop through all tokens in the sentence
        for idx, (token_start, token_end) in enumerate(encoding.offset_mapping):
            if token_start == start:
                labels[idx] = f"B-{ent_label}"
            if token_start > start and token_end <= end:
                labels[idx] = f"I-{ent_label}"

    # convert the labels to ids and return
    label_ids = [label_to_id.get(label, label_to_id["O"]) for label in labels]

    return encoding["input_ids"], encoding["attention_mask"], label_ids

In [17]:
# load the annotated data in json format
with open("../01_data/annotations.json", "r") as f:
    data = json.load(f)

# initialize label dictionary
label_dict = {"O"}

# loop through all sentences
for task in data:
    # loop through all annotations per sentence
    for result in task["annotations"][0]["result"]:
        # check if annoation is of type label
        if result["type"] == "labels":
            # check if there is actually only one label per annotation
            num_labels = len(result["value"]["labels"])
            if num_labels > 1:
                print("More than one label assigned")
            # get the label (only social group, not the sentiment) and add to the dictionary
            label = result["value"]["labels"][0][0:2]
            label_dict.add(f"B-{label}")
            label_dict.add(f"I-{label}")

# sort the label dictionary
label_list = sorted(label_dict)

# create label to id dictionary based on sorted list
label_to_id = {label: i for i, label in enumerate(label_list)}


In [43]:
# initialize empty dataset list
dataset = []

# loop through all sentences in the data
for task in data:
    # get the sentence and all annotations
    text = task["data"]["text"]
    results = task["annotations"][0]["result"]
    # store annotation spans in a list
    spans = [
        {
            "start": r["value"]["start"],
            "end": r["value"]["end"],
            "labels": r["value"]["labels"][0][0:2]
        }
        for r in results if r["type"] == "labels"
    ]
    # tokenize and get label ids
    input_ids, attention_mask, label_ids = tokenize_and_align_labels(text, spans, label_to_id)
    # add everything to the dataset list
    dataset.append({
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_ids,
        "text": text
    })

In [49]:
# split into training and test dataset
split_idx = int(len(dataset) * 0.75)
train_dataset = dataset[:split_idx]
test_dataset = dataset[split_idx:]

In [ ]:
# define functon to create tensor dataset
def create_tensor_dataset(dataset):
    input_ids_list = [item["input_ids"] for item in dataset]
    attention_masks_list = [item["attention_mask"] for item in dataset]
    label_ids_list = [item["labels"] for item in dataset]

    max_len = max(len(seq) for seq in input_ids_list)

    input_ids = torch.tensor([seq + [1]*(max_len - len(seq)) for seq in input_ids_list])
    attention_masks = torch.tensor([seq + [0]*(max_len - len(seq)) for seq in attention_masks_list])
    labels = torch.tensor([[-100] + seq[1:-1] + [-100] + [-100]*(max_len - len(seq)) for seq in label_ids_list])

    tensor_dataset = TensorDataset(input_ids, attention_masks, labels)

    return tensor_dataset

# create proper train and test tensor datasets
train_dataset = create_tensor_dataset(train_dataset)
test_dataset = create_tensor_dataset(test_dataset)

# turn them into dataloaders for batch training
train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=5)
test_dataloader = DataLoader(test_dataset, sampler=SequentialSampler(test_dataset), batch_size=5)
    

In [53]:
# create id to label dictionary
id_to_label = {v: k for k, v in label_to_id.items()}

# model setup
model = RobertaForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label_to_id),
    id2label=id_to_label,
    label2id=label_to_id
)

# use gpu if available, otherwise cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# define optimizer, learning rate and the number of epochs
optimizer = AdamW(model.parameters(), lr=1e-5)
epochs = 10

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [54]:
# set the model to training mode
model.train()

# loop through epochs
for epoch in range(epochs):

    # print the epoch number
    print(f"Epoch {epoch + 1}/{epochs}")

    # initialize training loss for the epoch
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc="Training")

    # loop through each batch
    for batch in progress_bar:

        # move all batch data to respective device
        input_ids, attention_masks, labels = [b.to(device) for b in batch]

        # clear the old gradient
        model.zero_grad()

        # run data through the model and save the outputs
        outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)

        # save the loss and add to the total loss for the epoch
        loss = outputs.loss
        total_loss += loss.item()

        # compute gradients by backpropagation, cap gradients to prevent gradient explosion
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)

        # update the model weights based on the gradient and update the progress bar
        optimizer.step()
        progress_bar.set_postfix(loss=loss.item())

    # get the average training loss per batch and print
    avg_loss = total_loss / len(train_dataloader)
    print(f"Average training loss: {avg_loss:.4f}")


Epoch 1/10


Training: 100%|██████████| 15/15 [00:09<00:00,  1.54it/s, loss=0.419]


Average training loss: 0.7727
Epoch 2/10


Training: 100%|██████████| 15/15 [00:08<00:00,  1.73it/s, loss=0.208]


Average training loss: 0.2120
Epoch 3/10


...

Average training loss: 0.0264
Epoch 9/10


Training: 100%|██████████| 15/15 [00:09<00:00,  1.61it/s, loss=0.00255]


Average training loss: 0.0143
Epoch 10/10


Training: 100%|██████████| 15/15 [00:08<00:00,  1.69it/s, loss=0.00358]

Average training loss: 0.0098


In [96]:
# set the model to evaluation mode (no loss calculation)
model.eval()

# initialize empty lists for true and predicted labels
true_labels = []
pred_labels = []

# proceed without calculating gradients
with torch.no_grad():

    # loop through all batches in the test dataloader
    for batch in test_dataloader:

        # move inputs to respective device
        input_ids, attention_masks, labels = [b.to(device) for b in batch]

        # get model outputs, get logits and get the class labels for the max logit
        outputs = model(input_ids=input_ids, attention_mask=attention_masks)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=2)

        # loop through all labels for all sentences in the batch
        for i in range(len(labels)):
            
            # get the true and the predicted label
            true_seq = labels[i].cpu().numpy()
            pred_seq = predictions[i].cpu().numpy()

            # append true and predicted label only if the true label is not -100 (special token)
            for t, p in zip(true_seq, pred_seq):
                if t != -100:
                    true_labels.append(t)
                    pred_labels.append(p)

# print the classification report
print(classification_report(
    true_labels,
    pred_labels
))

              precision    recall  f1-score   support

           0       0.70      0.64      0.67        11
           1       0.83      0.38      0.53        13
           2       0.99      1.00      0.99       720

    accuracy                           0.98       744
   macro avg       0.84      0.67      0.73       744
weighted avg       0.98      0.98      0.98       744



In [58]:
# define function for predicting a specific sentence
def predict_sentence(sentence, model, tokenizer, id_to_label, device='cpu'):

    # set model to evaluation mode
    model.eval()

    # tokenize the input sentence, return PyTorch tensors and offset mappings
    encoding = tokenizer(
        sentence,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True
    )

    # extract input ids, attention mask and offset mapping
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    offset_mapping = encoding["offset_mapping"][0]

    # without computing gradients run the input through the trained model
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)[0]

    # get the tokens and all predicted labels for the sentence
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    labels = [id_to_label[pred.item()] for pred in predictions]

    # combine tokens and labels, ignoring special tokens (CLS and end of sequence)
    result = []
    for token, label, (start, end) in zip(tokens, labels, offset_mapping):
        if start == 0 and end == 0:
            continue
        token_text = sentence[start:end]
        result.append((token_text, label))

    return result

In [95]:
# try out a custom sentence
predict_sentence("Our party supports the rights of business people.", model=model, tokenizer=tokenizer, id_to_label=id_to_label)

[('Our', 'O'),
 ('party', 'O'),
 ('supports', 'O'),
 ('the', 'O'),
 ('rights', 'O'),
 ('of', 'O'),
 ('business', 'B-sg'),
 ('people', 'I-sg'),
 ('.', 'O')]